# 🦜🔗 LangChain Crash Course
## *Build AI Apps That Actually Do Things*

---

> **Who this is for:** Anyone curious about building LLM-powered apps — no ML background needed.  
> **What you'll build:** A smart study assistant that searches the web, remembers conversations, and answers questions about any topic.

### 🗺️ What We'll Cover
| # | Topic | What You'll Build |
|---|-------|-------------------|
| 1 | **Setup & Basics** | Your first LLM call |
| 2 | **Prompt Templates** | Reusable, dynamic prompts |
| 3 | **Chains** | Multi-step pipelines |
| 4 | **Memory** | Chatbot that remembers you |
| 5 | **Tools** | Give AI calculator + web search |
| 6 | **Agents** | AI that decides what to do |
| 7 | **🎓 Capstone** | Full study assistant |

---
*Uses **OpenAI** (gpt-3.5-turbo) — cheap, fast, and great for learning. Swap in any model you like.*

---
## 📦 Section 0 — Install & Setup

Run this once. It installs everything we need.

In [1]:
# Install all required packages
# This takes ~30 seconds
!pip install -q langchain langchain-openai langchain-community openai duckduckgo-search


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
!pip install -U ddgs

   ---------------------------------------- 0.0/70.6 kB ? eta -:--:--
   ---------------------------------------- 70.6/70.6 kB 3.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/161.7 kB ? eta -:--:--
   ---------------------------------------- 161.7/161.7 kB 9.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/61.8 kB ? eta -:--:--
   ---------------------------------------- 61.8/61.8 kB ? eta 0:00:00
   ---------------------------------------- 0.0/369.0 kB ? eta -:--:--
   --------------------------------------- 369.0/369.0 kB 22.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import importlib.metadata
import langchain
import openai

print(f"langchain: {langchain.__version__}")
print(f"openai: {openai.__version__}")

# Safe way to check packages without internal __version__ attributes
print(f"langchain_openai: {importlib.metadata.version('langchain-openai')}")
print(f"duckduckgo_search: {importlib.metadata.version('duckduckgo-search')}")


langchain: 1.3.6
openai: 2.36.0
langchain_openai: 1.2.2
duckduckgo_search: 8.1.1


In [ ]:
import os

# ─────────────────────────────────────────────────────
# 🔑  PASTE YOUR OPENAI API KEY HERE
# Get one free at: https://platform.openai.com/api-keys
# ─────────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "dsjljls"   # ← replace this

# Quick sanity check
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
print("✅ LLM connected! Model:", llm.model_name)

✅ LLM connected! Model: gpt-3.5-turbo


---
## 🌱 Section 1 — LangChain Basics

### What is LangChain?
Think of LangChain as **LEGO for AI apps**.  
Each piece (LLM, prompt, chain, tool) snaps together.

```
Your Question
     ↓
Prompt Template  (formats the question nicely)
     ↓
LLM             (OpenAI / Claude / Llama etc.)
     ↓
Output Parser   (turns raw text into structured data)
     ↓
Your Answer ✨
```

### The 3 Core Objects
| Object | What it does | Real-world analogy |
|--------|-------------|-------------------|
| `ChatOpenAI` | Talks to the model | Your "AI brain" |
| `HumanMessage` | Your message | A text to a friend |
| `AIMessage` | Model's reply | Friend's reply |

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)

# Most basic usage — just send a message
response = llm.invoke("What is photosynthesis in one sentence?")

Type: <class 'langchain_core.messages.ai.AIMessage'>
Content: Photosynthesis is the process by which plants, algae, and some bacteria convert sunlight, carbon dioxide, and water into glucose and oxygen.


In [8]:
print("Type:", type(response))
print("------------------------------")
print("response:", response)
print("------------------------------")
print("Content:", response.content)

Type: <class 'langchain_core.messages.ai.AIMessage'>
------------------------------
response: content='Photosynthesis is the process by which plants, algae, and some bacteria convert sunlight, carbon dioxide, and water into glucose and oxygen.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 15, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dp8MHKpw4vjl4RfOXQo18u1y1diTC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019eb098-3bbe-7bb1-a6b2-2250dd821611-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 27, 'total_tokens': 42, 'input_token_details': {'audio': 0, '

In [9]:
# Add a SystemMessage to set the AI's persona
messages = [
    SystemMessage(content="You are a funny science teacher who explains things using pizza analogies."),
    HumanMessage(content="Explain how DNA replication works.")
]

response = llm.invoke(messages)
print(response.content)

Sure thing! Think of DNA replication like making a pizza. 

First, imagine your original pizza (the parent DNA molecule) sitting on the kitchen counter. Now, the first step in DNA replication is to "unzip" the two strands of the DNA molecule, just like separating the two halves of your pizza crust.

Next, imagine that each unzipped strand serves as a template for building a new strand. Enzymes called DNA polymerases come in and add complementary nucleotides to each template strand, just like adding toppings to each half of your pizza crust.

As the DNA polymerases add nucleotides, they ensure that the new DNA strands match up perfectly with the original strands, just like making sure your pizza toppings are perfectly aligned on each half of the pizza.

Finally, you end up with two identical DNA molecules, just like having two identical pizzas, each made from one original pizza. And that's how DNA replication works, using the delicious analogy of making a pizza!


### 🧪 Try It Yourself
Change the SystemMessage persona below and ask anything!
Common ideas:
- `"You are a pirate who loves math"`
- `"You are a 5-year-old explaining complex topics"`
- `"You are Gordon Ramsay explaining programming concepts"`

In [10]:
# ✏️ YOUR TURN: Change the persona and question!
messages = [
    SystemMessage(content="You are a sports commentator explaining computer science concepts."),
    HumanMessage(content="Explain what a for-loop is.")
]
response = llm.invoke(messages)
print(response.content)

In computer science, a for-loop is a programming construct that allows you to repeat a block of code a specific number of times. It consists of three main components: an initialization step, a condition to check before each iteration, and an update step. 

Here's an analogy to help understand a for-loop in a sports context: Imagine a basketball player practicing free throws. The player decides to take 10 shots. In this scenario, the player is essentially using a for-loop to repeat the action of taking a shot 10 times. 

So, the initialization step would be setting the starting point (the first shot), the condition would be checking if the player has taken 10 shots yet, and the update step would be moving on to the next shot after each attempt. This structure allows the player to efficiently practice and improve their skills without having to manually repeat the same action multiple times.


---
## 📝 Section 2 — Prompt Templates

### Why Templates?

Imagine writing the same email over and over, just changing the name.  
You'd use a template: *"Dear {name}, thank you for..."*

Prompt Templates work the same way — **reusable prompts with variables**.

```python
# Without template (repetitive 😩)
"Explain photosynthesis to a 10-year-old in 2 sentences"
"Explain gravity to a 10-year-old in 2 sentences"
"Explain evolution to a 10-year-old in 2 sentences"

# With template (clean ✨)
"Explain {topic} to a {age}-year-old in {sentences} sentences"
```

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# Define a reusable template with {variables}
explain_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful tutor who explains things clearly and concisely."),
    ("human", "Explain {topic} to someone who is {level}. Keep it under {max_sentences} sentences.")
])

# Render the template with different values — no LLM call yet
rendered = explain_template.format_messages(
    topic="black holes",
    level="in middle school",
    max_sentences=3
)

print("--- Rendered Prompt ---")
for msg in rendered:
    print(f"[{msg.type.upper()}]: {msg.content}")

--- Rendered Prompt ---
[SYSTEM]: You are a helpful tutor who explains things clearly and concisely.
[HUMAN]: Explain black holes to someone who is in middle school. Keep it under 3 sentences.


In [12]:
# Now use it with the LLM — try different topics!
topics_to_explain = [
    {"topic": "machine learning", "level": "a complete beginner", "max_sentences": 3},
    {"topic": "cryptocurrency",   "level": "a skeptical parent",   "max_sentences": 4},
    {"topic": "quantum computing","level": "a high school student","max_sentences": 3},
]

for params in topics_to_explain:
    messages = explain_template.format_messages(**params)
    response = llm.invoke(messages)
    print(f"\n🔷 {params['topic'].upper()} (for {params['level']}):")
    print(response.content)
    print("-" * 50)


🔷 MACHINE LEARNING (for a complete beginner):
Machine learning is a branch of artificial intelligence where computer systems are trained to learn patterns from data and make decisions or predictions without being explicitly programmed. It involves algorithms that improve their performance over time as they are exposed to more data. The goal is to enable machines to learn and adapt on their own to perform tasks more accurately and efficiently.
--------------------------------------------------

🔷 CRYPTOCURRENCY (for a skeptical parent):
Cryptocurrency is a digital form of money that uses encryption techniques to regulate the generation of units and verify the transfer of funds. It operates independently of a central authority, like a government or bank, through a decentralized network called blockchain. While it can offer benefits such as lower transaction fees and faster transfers, it also carries risks due to its volatile nature and potential for misuse in illegal activities. It's im

In [13]:
# FewShot Prompting — teach with examples
from langchain_core.prompts import FewShotChatMessagePromptTemplate

# Show the model examples of what you want
examples = [
    {"input": "happy", "output": "joyful 😄"},
    {"input": "sad",   "output": "melancholic 😢"},
    {"input": "fast",  "output": "lightning-quick ⚡"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "Give me a fancier synonym with emoji for: {input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a vocabulary enhancer."),
    few_shot_prompt,
    ("human", "Give me a fancier synonym with emoji for: {input}"),
])

# ✏️ Try different words!
for word in ["boring", "big", "scared", "smart"]:
    response = llm.invoke(final_prompt.format_messages(input=word))
    print(f"  {word:10s} → {response.content}")

  boring     → tedious 😴
  big        → colossal 🌟
  scared     → terrified 😱
  smart      → astute 🧠


---
## ⛓️ Section 3 — Chains

### What's a Chain?

A chain connects steps together like an **assembly line**.

```
Raw Input
    ↓
Step 1: Prompt Template  (format the question)
    ↓
Step 2: LLM             (think about it)
    ↓  
Step 3: Output Parser   (clean up the answer)
    ↓
Final Output ✅
```

LangChain uses the `|` pipe operator — just like Unix!  
`prompt | llm | parser` reads left to right. Beautiful.

### Types of Chains
| Chain | Use Case |
|-------|----------|
| `prompt \| llm` | Basic: prompt → answer |
| `prompt \| llm \| parser` | Structured output |
| Sequential | Chain A's output → Chain B's input |
| Parallel | Run multiple chains at once |

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# ── Chain 1: Simple Q&A Chain 
qa_prompt = ChatPromptTemplate.from_template(
    "Answer this question in exactly 2 sentences: {question}"
)

# The | pipe creates a chain (called LCEL — LangChain Expression Language)
qa_chain = qa_prompt | llm | StrOutputParser()

# Run it!
answer = qa_chain.invoke({"question": "Why is the sky blue?"})
print("Answer:", answer)
print("Type:", type(answer))  # Notice: string, not AIMessage!

Answer: The sky appears blue because of the way the Earth's atmosphere scatters sunlight. Shorter blue wavelengths are scattered more efficiently than longer red wavelengths, causing the blue light to dominate our perception of the sky's color.
Type: <class 'langchain_core.messages.base.TextAccessor'>


In [15]:
# ── Chain 2: Sequential Chain 
# Step 1: Explain a concept
# Step 2: Create a quiz question about it
# Output of step 1 feeds INTO step 2

from langchain_core.runnables import RunnablePassthrough

explain_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in 3 clear bullet points for a beginner."
)

quiz_prompt = ChatPromptTemplate.from_template(
    """Based on this explanation:

{explanation}

Create ONE multiple choice question with 4 options (A/B/C/D) and mark the correct answer.
Format it nicely."""
)

# Build the pipeline
explain_chain = explain_prompt | llm | StrOutputParser()

# Chain them: explanation feeds into quiz creation
full_pipeline = (
    {"explanation": explain_chain, "topic": RunnablePassthrough()}
    | quiz_prompt
    | llm
    | StrOutputParser()
)

# Run the whole pipeline with just a topic!
result = full_pipeline.invoke({"topic": "blockchain technology"})
print(result)

What is blockchain primarily used for?
A) Centralizing digital transactions
B) Increasing the need for intermediaries
C) Enhancing security and transparency
D) Decreasing the permanency of records

Correct answer: C) Enhancing security and transparency


In [16]:
# ── Chain 3: Parallel Chain ────────────────────────────────────────
# Run multiple chains at the SAME TIME and combine results
from langchain_core.runnables import RunnableParallel

topic_input = ChatPromptTemplate.from_template("Topic: {topic}")

pros_chain = (
    ChatPromptTemplate.from_template("List 3 pros of {topic}. Be brief.")
    | llm | StrOutputParser()
)

cons_chain = (
    ChatPromptTemplate.from_template("List 3 cons of {topic}. Be brief.")
    | llm | StrOutputParser()
)

summary_chain = (
    ChatPromptTemplate.from_template("Summarize {topic} in one sentence.")
    | llm | StrOutputParser()
)

# All three run simultaneously!
parallel_chain = RunnableParallel(
    pros=pros_chain,
    cons=cons_chain,
    summary=summary_chain
)

results = parallel_chain.invoke({"topic": "social media"})

print("📋 SUMMARY:")
print(results["summary"])
print("\n✅ PROS:")
print(results["pros"])
print("\n❌ CONS:")
print(results["cons"])

📋 SUMMARY:
Social media is a digital platform that allows users to connect and interact with others, share content, and stay informed on news and trends.

✅ PROS:
1. Connectivity: Social media allows people to stay connected with friends and family, no matter where they are in the world.
2. Information sharing: Social media platforms provide a quick and easy way to share news, updates, and information with a large audience.
3. Networking: Social media can be a valuable tool for networking and building professional relationships, connecting individuals with potential employers or collaborators.

❌ CONS:
1. Negative impact on mental health - exposure to unrealistic standards, cyberbullying, and addiction.
2. Privacy concerns - data breaches, surveillance, and lack of control over personal information.
3. Spread of misinformation - fake news, propaganda, and echo chambers leading to increased polarization.


In [18]:
# ── Chain 4: Structured Output Chain 
# Make the LLM return JSON that you can use in your code
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import List

# FIX: Import directly from standard pydantic instead of the deprecated v1 bridge
from pydantic import BaseModel, Field

# Define the exact shape of data you want back
class MovieRecommendation(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    why: str = Field(description="Why someone would like it, one sentence")
    rating: float = Field(description="Your rating out of 10")

class MovieList(BaseModel):
    movies: List[MovieRecommendation] = Field(description="List of 3 movies")

# Set up the parser with your schema
parser = JsonOutputParser(pydantic_object=MovieList)

# FIX: Added prompt template import above and formatted instructions
movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a movie critic. Always respond with valid JSON."),
    ("human", "Recommend 3 movies for someone who likes {genre}. {format_instructions}"),
])

# Ensure your model is instantiated (e.g., llm = ChatOpenAI(model="gpt-4o", temperature=0))
movie_chain = movie_prompt | llm | parser

result = movie_chain.invoke({
    "genre": "mind-bending sci-fi",
    "format_instructions": parser.get_format_instructions()
})

# Now it's a real Python dict!
print("Type:", type(result))
for movie in result["movies"]:
    print(f"\n🎬 {movie['title']} ({movie['year']}) — ⭐ {movie['rating']}/10")
    print(f"   {movie['why']}")


Type: <class 'dict'>

🎬 Inception (2010) — ⭐ 9.5/10
   Mind-bending narrative within dreams

🎬 Blade Runner 2049 (2017) — ⭐ 9.0/10
   Stunning visuals and philosophical themes

🎬 Arrival (2016) — ⭐ 9.2/10
   Unique take on language and time perception


---
## 🧠 Section 4 — Memory

### The Problem with LLMs

LLMs have **no memory** by default.  
Every call is a fresh start — like talking to someone with amnesia.

```python
llm.invoke("My name is Alice")          # "Nice to meet you, Alice!"
llm.invoke("What's my name?")           # "I don't know your name..."  😕
```

### The Solution: Store the history!

LangChain gives us memory modules that **automatically track the conversation**.

| Memory Type | How it works | Best for |
|-------------|-------------|----------|
| `ChatMessageHistory` | Keeps ALL messages | Short conversations |
| `ConversationBufferWindowMemory` | Keeps last N messages | Longer chats |
| `ConversationSummaryMemory` | Summarizes old messages | Very long sessions |

In [19]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# ── Step 1: The prompt with a {history} placeholder ────────────────
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly study buddy named Alex. "
               "You remember everything the student tells you about themselves."),
    MessagesPlaceholder(variable_name="history"),   # ← history goes here
    ("human", "{input}"),
])

# ── Step 2: Base chain (no memory yet) ─────────────────────────────
base_chain = chat_prompt | llm | StrOutputParser()

# ── Step 3: In-memory store ─────────────────────────────────────────
store = {}  # session_id → ChatMessageHistory

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# ── Step 4: Wrap with memory ────────────────────────────────────────
chatbot = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("✅ Memory chatbot ready!")

✅ Memory chatbot ready!


C:\Users\hi\Desktop\projects\python_projects\tutorial\play_langchain_llamaindex_langgraph\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
# Simulate a real conversation
config = {"configurable": {"session_id": "student_001"}}

conversation = [
    "Hi! I'm Priya, I'm studying for my biology exam tomorrow.",
    "I'm really struggling with the concept of mitosis.",
    "Can you give me a quick memory trick to remember the phases?",
    "What was the topic I said I'm struggling with?",   # ← tests memory!
    "And what's my name again?",                         # ← tests memory!
]

for message in conversation:
    print(f"\n👤 Student: {message}")
    response = chatbot.invoke({"input": message}, config=config)
    print(f"🤖 Alex: {response}")
    print("-" * 60)


👤 Student: Hi! I'm Priya, I'm studying for my biology exam tomorrow.
🤖 Alex: Hi Priya! Nice to meet you. I'm here to help you with your biology exam preparation. What topics are you focusing on for the exam?
------------------------------------------------------------

👤 Student: I'm really struggling with the concept of mitosis.
🤖 Alex: No problem, Priya! Mitosis can be a tricky concept, but I'm sure we can work through it together. Let's start by reviewing the different stages of mitosis. Do you have any specific questions about it?
------------------------------------------------------------

👤 Student: Can you give me a quick memory trick to remember the phases?
🤖 Alex: Of course, Priya! One common memory trick to remember the phases of mitosis is the phrase "PMAT" - Prophase, Metaphase, Anaphase, Telophase. You can also create a sentence using the first letter of each phase to help you remember the order. How about "Please Make A Taco"? Does that help you remember the phases bett

In [21]:
# Inspect what's stored in memory
print("\n📚 Full conversation history:")
history = store["student_001"]
for i, msg in enumerate(history.messages, 1):
    role = "👤 Human" if msg.type == "human" else "🤖 AI"
    preview = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"{i}. {role}: {preview}")

print(f"\nTotal messages stored: {len(history.messages)}")


📚 Full conversation history:
1. 👤 Human: Hi! I'm Priya, I'm studying for my biology exam tomorrow.
2. 🤖 AI: Hi Priya! Nice to meet you. I'm here to help you with your biology exam preparat...
3. 👤 Human: I'm really struggling with the concept of mitosis.
4. 🤖 AI: No problem, Priya! Mitosis can be a tricky concept, but I'm sure we can work thr...
5. 👤 Human: Can you give me a quick memory trick to remember the phases?
6. 🤖 AI: Of course, Priya! One common memory trick to remember the phases of mitosis is t...
7. 👤 Human: What was the topic I said I'm struggling with?
8. 🤖 AI: You mentioned that you're struggling with the concept of mitosis, Priya. Is ther...
9. 👤 Human: And what's my name again?
10. 🤖 AI: Your name is Priya. It's nice to chat with you and help you with your biology ex...

Total messages stored: 10


In [22]:
# Multiple students — each has their OWN memory!
students = {
    "student_A": "Hi, I'm Raj and I love quantum physics.",
    "student_B": "Hey, I'm Maria and I'm terrible at math.",
}

for student_id, intro in students.items():
    config = {"configurable": {"session_id": student_id}}
    chatbot.invoke({"input": intro}, config=config)

# Ask both the same question — different answers!
question = "What subject did I mention?"
for student_id in students:
    config = {"configurable": {"session_id": student_id}}
    response = chatbot.invoke({"input": question}, config=config)
    print(f"\n[{student_id}] → {response}")


[student_A] → You mentioned that you love quantum physics, Raj! It's awesome that you have a passion for such a complex and interesting field of science.

[student_B] → You mentioned that you're struggling with math, Maria. Don't worry, we'll tackle those math problems together!


---
## 🔧 Section 5 — Tools

### LLMs Are Powerful But Limited

| LLM Can ✅ | LLM Can't ❌ |
|-----------|------------|
| Reason about anything | Know today's news |
| Explain complex topics | Do precise math |
| Write and summarize | Access the internet |
| Translate languages | Check real-time data |

**Tools** give the LLM superpowers by letting it call external functions.

### How It Works
```
User: "What's 2847 × 392?"
          ↓
LLM thinks: "I should use the calculator tool"
          ↓
Calls: calculator(2847 × 392)
          ↓
Gets: 1,115,624
          ↓
LLM responds: "2847 × 392 = 1,115,624"
```

You can make ANY Python function into a LangChain tool with `@tool`!

In [42]:
from langchain_core.tools import tool
import math
import datetime
import random

# ── Tool 1: Calculator 
@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. 
    Input should be a valid Python math expression like '2 + 2' or 'math.sqrt(16)'.
    Use this for ANY calculation."""
    try:
        # Safe eval with only math functions allowed
        result = eval(expression, {"__builtins__": {}}, {"math": math, **vars(math)})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

# ── Tool 2: Get Today's Date 
@tool
def get_current_date() -> str:
    """Get today's date and time. Use this when asked about current date or time."""
    now = datetime.datetime.now()
    return f"Current date: {now.strftime('%A, %B %d, %Y')} | Time: {now.strftime('%I:%M %p')}"

# ── Tool 3: Random Fact Generator 
@tool  
def random_study_tip() -> str:
    """Get a random study tip. Use when student seems stressed or asks for study advice."""
    tips = [
        "🍅 Try the Pomodoro Technique: 25 min focus, 5 min break!",
        "✍️  Teach it to explain it — the Feynman Technique works wonders.",
        "😴 Sleep is when your brain consolidates memories. Don't skip it!",
        "🏃 A 10-minute walk boosts memory and focus for hours.",
        "📝 Handwriting notes beats typing for long-term retention.",
        "🔄 Spaced repetition: review today, tomorrow, next week, next month.",
    ]
    return random.choice(tips)

# Test the tools directly first!
print(calculator.invoke({"expression": "math.pi * 15**2"}))
print(get_current_date.invoke({}))
print(random_study_tip.invoke({}))

Result: 706.8583470577034
Current date: Wednesday, June 10, 2026 | Time: 02:39 PM
🍅 Try the Pomodoro Technique: 25 min focus, 5 min break!


In [43]:
# ── Tool 4: Web Search (DuckDuckGo — FREE, no API key!) ──────────────
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

# Test it
result = search.invoke("latest breakthrough in quantum computing 2024")
print(result[:500] + "...")

February 24, 2026 - Better qubits only matter if we can use them on real problems. In 2024, many research groups focused on hybrid approaches that mix classical computing, AI and quantum hardware so each part handles the work it does best. March 8, 2026 - In 2024, researchers made major progress in quantum error correction and qubit design. New qubit types are more stable. Scientists are also improving topological qubits, which naturally resist noise. February 2, 2026 - In 2024, researchers crea...


In [44]:
# ── Tool 5: Unit Converter
@tool
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between common units. 
    Supports: km/miles, kg/pounds, celsius/fahrenheit, liters/gallons.
    Example: unit_converter(100, 'km', 'miles')"""
    conversions = {
        ("km", "miles"):       lambda x: x * 0.621371,
        ("miles", "km"):       lambda x: x * 1.60934,
        ("kg", "pounds"):      lambda x: x * 2.20462,
        ("pounds", "kg"):      lambda x: x * 0.453592,
        ("celsius", "fahrenheit"): lambda x: x * 9/5 + 32,
        ("fahrenheit", "celsius"): lambda x: (x - 32) * 5/9,
        ("liters", "gallons"): lambda x: x * 0.264172,
        ("gallons", "liters"): lambda x: x * 3.78541,
    }
    key = (from_unit.lower(), to_unit.lower())
    if key in conversions:
        result = conversions[key](value)
        return f"{value} {from_unit} = {result:.4f} {to_unit}"
    return f"Conversion from {from_unit} to {to_unit} not supported."

# Test it
print(unit_converter.invoke({"value": 100, "from_unit": "km", "to_unit": "miles"}))
print(unit_converter.invoke({"value": 37, "from_unit": "celsius", "to_unit": "fahrenheit"}))

# Show tool metadata — LangChain reads these!
print("\n--- Tool Metadata ---")
print("Name:", calculator.name)
print("Description:", calculator.description[:80])

100.0 km = 62.1371 miles
37.0 celsius = 98.6000 fahrenheit

--- Tool Metadata ---
Name: calculator
Description: Evaluate a mathematical expression. 
    Input should be a valid Python math exp


---
## 🤖 Section 6 — Agents

### From Chains to Agents

| | Chain | Agent |
|--|-------|-------|
| **Flow** | Fixed, predetermined | Dynamic, decided by LLM |
| **Tools** | Hardcoded | Chosen as needed |
| **Steps** | Always the same | Varies by problem |
| **Analogy** | Recipe | Chef improvising |

### The ReAct Loop

Agents use **ReAct** (Reason + Act) to think step by step:

```
Question: "What's the population of Tokyo divided by the population of Paris?"

🧠 Thought: I need to find both populations first.
🔧 Action: search("population of Tokyo")
👁️  Observation: Tokyo population is ~14 million
🧠 Thought: Now I need Paris.
🔧 Action: search("population of Paris")  
👁️  Observation: Paris population is ~2.1 million
🧠 Thought: Now I can calculate 14M / 2.1M
🔧 Action: calculator("14000000 / 2100000")
👁️  Observation: 6.67
🧠 Thought: I have the answer!
💬 Final Answer: Tokyo's population is about 6.7x larger than Paris.
```

In [45]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── Step 1: Same tools, no change ──────────────────────────────────
agent_tools = [
    calculator,
    get_current_date,
    random_study_tip,
    unit_converter,
    search,
]

# ── Step 2: Simpler prompt — no agent_scratchpad needed ────────────
# system_prompt is now just a plain string
system_prompt = """You are a knowledgeable and friendly study assistant.

You have access to tools — USE THEM when needed:
- calculator: for any math
- get_current_date: for today's date
- random_study_tip: when students need motivation
- unit_converter: for unit conversions
- duckduckgo_search: for looking up current information

Always explain your reasoning. Show your work."""

# ── Step 3: Create agent directly 
agent = create_agent(
    model=llm,
    tools=agent_tools,
    system_prompt=system_prompt,
)

print("✅ Agent ready!")

✅ Agent ready!


In [46]:
# ── Watch the agent THINK and USE TOOLS ───────────────────────────
# verbose=True shows the ReAct loop in action

print("=" * 60)
print("QUESTION: Calculate the area of a circle with radius 7cm,")
print("          then convert it to square inches.")
print("=" * 60)

result = agent.invoke({
    "messages": [{"role": "user", 
                  "content": "Calculate the area of a circle with radius 7cm, then convert to square inches."
                 }]
})

print("\n🏁 FINAL ANSWER:")
print(result["messages"][-1].content)

QUESTION: Calculate the area of a circle with radius 7cm,
          then convert it to square inches.

🏁 FINAL ANSWER:
The area of the circle with a radius of 7 cm is approximately \(23.85 \, \text{square inches}\) when converted.


In [47]:
result

{'messages': [HumanMessage(content='Calculate the area of a circle with radius 7cm, then convert to square inches.', additional_kwargs={}, response_metadata={}, id='c1d28e5e-d837-41f1-90e5-aad7044e688c'),
  AIMessage(content="To calculate the area of a circle with radius \\( r = 7 \\, \\text{cm} \\), we use the formula for the area of a circle:\n\n\\[ \\text{Area} = \\pi \\times r^2 \\]\n\nSubstitute the radius \\( r = 7 \\, \\text{cm} \\) into the formula:\n\n\\[ \\text{Area} = \\pi \\times 7^2 \\]\n\nLet's calculate the area first.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 340, 'total_tokens': 458, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Dp9

In [48]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

print("🧐 --- AGENT EXECUTION LOG ---\n")

for msg in result['messages']:
    if isinstance(msg, HumanMessage):
        print(f"👤 User Input:\n   {msg.content}\n")
        
    elif isinstance(msg, AIMessage):
        # FIX: Clean the text outside of the f-string to avoid backslash errors
        text_content = msg.content.strip().replace('\n', '\n   ')
        
        if text_content:
            print(f"🤖 Assistant Thought:\n   {text_content}")
        
        if msg.tool_calls:
            for t in msg.tool_calls:
                print(f"🔧 Tool Call -> Name: [{t['name']}]")
                print(f"   Arguments: {t['args']}")
        print() 
        
    elif isinstance(msg, ToolMessage):
        print(f"📥 Tool Output [{msg.name}]:\n   {msg.content}\n")
        print("─" * 50 + "\n")


🧐 --- AGENT EXECUTION LOG ---

👤 User Input:
   Calculate the area of a circle with radius 7cm, then convert to square inches.

🤖 Assistant Thought:
   To calculate the area of a circle with radius \( r = 7 \, \text{cm} \), we use the formula for the area of a circle:
   
   \[ \text{Area} = \pi \times r^2 \]
   
   Substitute the radius \( r = 7 \, \text{cm} \) into the formula:
   
   \[ \text{Area} = \pi \times 7^2 \]
   
   Let's calculate the area first.
🔧 Tool Call -> Name: [calculator]
   Arguments: {'expression': '3.14 * 7**2'}

📥 Tool Output [calculator]:
   Result: 153.86

──────────────────────────────────────────────────

🤖 Assistant Thought:
   The area of the circle with a radius of 7 cm is \(153.86 \, \text{cm}^2\).
   
   Now, let's convert this area to square inches. 
   
   To convert square centimeters to square inches, we use the conversion factor:
   1 square inch = 6.4516 square centimeters
   
   Let's calculate the area in square inches.
🔧 Tool Call -> Name: [ca

In [49]:
# ── Agent using web search
print("=" * 60)
print("QUESTION: Search-based question")
print("=" * 60)

result = agent.invoke({
    "messages": [{"role": "user", 
                  "content": "What is the speed of light? Convert it from km/s to miles/s."
                 }]
})

print("\n🏁 FINAL ANSWER:")
print(result["messages"][-1].content)

QUESTION: Search-based question

🏁 FINAL ANSWER:
The speed of light is approximately 299,792,458 meters per second. 

Converting this speed from kilometers per second to miles per second, we get approximately 186,282,339.42 miles per second.


In [50]:
# ── Stream the agent's response in real-time 
print("\n🌊 STREAMING MODE — watch tokens arrive in real-time\n")
print("-" * 60)

for chunk in agent.stream({
    "messages": [{"role": "user", 
                  "content": "Give me a study tip and tell me what day of the week it is today."
                 }]
}):
    if "agent" in chunk:
        content = chunk["agent"]["messages"][-1].content
        if content:
            print(content, end="", flush=True)
    elif "tools" in chunk:
        for msg in chunk["tools"]["messages"]:
            print(f"\n🔧 [{msg.name}]: {msg.content}", end="", flush=True)

print("\n" + "-" * 60)


🌊 STREAMING MODE — watch tokens arrive in real-time

------------------------------------------------------------

🔧 [random_study_tip]: 🏃 A 10-minute walk boosts memory and focus for hours.
🔧 [get_current_date]: Current date: Wednesday, June 10, 2026 | Time: 02:40 PM
------------------------------------------------------------


---
## 🎓 Section 7 — Capstone: Full Study Assistant

### Putting It All Together

We'll build **StudyBot** — a complete AI study assistant that has:

| Feature | From Section |
|---------|-------------|
| 💬 Multi-turn memory | Memory (Section 4) |
| 🔧 Calculator + web search | Tools (Section 5) |
| 🤖 Intelligent tool use | Agents (Section 6) |
| 📋 Structured output | Chains (Section 3) |
| 🎨 Friendly persona | Prompts (Section 2) |

This is a **production-ready pattern** you can adapt for real apps!

In [51]:
from langchain.agents import create_agent
# RunnableWithMessageHistory and ChatMessageHistory not needed in 1.x
# memory is handled via messages list directly

# ── StudyBot Tools
@tool
def create_quiz(topic: str, num_questions: int = 3) -> str:
    """Create a mini quiz on any topic. 
    Use this when a student wants to test their knowledge."""
    quiz_prompt = ChatPromptTemplate.from_template(
        "Create {n} multiple choice questions about {topic}. "
        "Format: Q1. [question] A) B) C) D) Answer: [letter]"
    )
    chain = quiz_prompt | llm | StrOutputParser()
    return chain.invoke({"topic": topic, "n": num_questions})

@tool
def explain_like_im_five(concept: str) -> str:
    """Explain any complex concept in very simple terms.
    Use when a student is confused or asks for a simpler explanation."""
    prompt = ChatPromptTemplate.from_template(
        "Explain {concept} using a simple analogy that a 10-year-old would understand. "
        "Use max 3 sentences and one everyday analogy."
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"concept": concept})

@tool
def make_study_plan(subject: str, days_available: int, hours_per_day: float) -> str:
    """Create a personalized study plan for an upcoming exam.
    Use when a student asks for help planning their study schedule."""
    prompt = ChatPromptTemplate.from_template(
        "Create a {days}-day study plan for {subject} with {hours} hours/day. "
        "Break it into daily topics with specific goals. Be practical."
    )
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"subject": subject, "days": days_available, "hours": hours_per_day})

studybot_tools = [
    calculator,
    get_current_date,
    random_study_tip,
    unit_converter,
    search,
    create_quiz,
    explain_like_im_five,
    make_study_plan,
]

print(f"✅ StudyBot has {len(studybot_tools)} tools:")
for t in studybot_tools:
    print(f"   🔧 {t.name}")

✅ StudyBot has 8 tools:
   🔧 calculator
   🔧 get_current_date
   🔧 random_study_tip
   🔧 unit_converter
   🔧 duckduckgo_search
   🔧 create_quiz
   🔧 explain_like_im_five
   🔧 make_study_plan


In [53]:
# ── Build StudyBot ───
studybot = create_agent(
    model=llm,
    tools=studybot_tools,
    system_prompt="""You are StudyBot 🎓 — a friendly, encouraging AI study companion.

Your personality:
- Warm and supportive (never make students feel dumb)
- Enthusiastic about learning
- Clear and concise in explanations
- Always celebrate progress

Your tools and when to use them:
- calculator → any math problem
- search → current events, facts you're unsure about
- explain_like_im_five → when student is confused
- create_quiz → when student wants to test themselves
- make_study_plan → when student needs organization help
- unit_converter → any unit conversion
- random_study_tip → when student seems overwhelmed
- get_current_date → scheduling questions

Remember the student's name, subjects, and goals throughout the conversation."""
)

# Memory: just a plain list we append to
chat_history = []

def chat(message, history=chat_history):
    history.append({"role": "user", "content": message})
    result = studybot.invoke({"messages": history})
    reply = result["messages"][-1].content
    history.append({"role": "assistant", "content": reply})
    return reply

print("🎓 StudyBot is ready!")

🎓 StudyBot is ready!


In [54]:
# ── Simulated Study Session ─────────────────────────────────────────
session_messages = [
    "Hi! I'm Aanya and I have a physics exam in 3 days on mechanics.",
    "Can you make me a 3-day study plan? I have 2 hours per day.",
    "I don't understand the difference between velocity and acceleration. Help!",
    "If a car accelerates from 0 to 60 mph in 4 seconds, what's its acceleration in m/s²?",
    "Quiz me on kinematics with 2 questions!",
    "I'm feeling overwhelmed... any advice?",
]

aanya_history = []   # separate history per student

for msg in session_messages:
    print(f"\n👩‍🎓 Aanya: {msg}")
    response = chat(msg, history=aanya_history)
    print(f"\n🤖 StudyBot: {response}")
    print("─" * 70)


👩‍🎓 Aanya: Hi! I'm Aanya and I have a physics exam in 3 days on mechanics.

🤖 StudyBot: Hello Aanya! 🌟 I'm here to help you prepare for your physics exam on mechanics. Let's start by creating a study plan to make sure you cover all the necessary topics. How many hours per day can you dedicate to studying for the next 3 days?
──────────────────────────────────────────────────────────────────────

👩‍🎓 Aanya: Can you make me a 3-day study plan? I have 2 hours per day.

🤖 StudyBot: Here's your 3-day study plan for your mechanics exam:

**Day 1:**
- Topic: Introduction to Mechanics
- Goals:
  - Review basic concepts such as force, motion, and Newton's laws of motion
  - Understand the difference between scalar and vector quantities
  - Solve simple problems involving motion and forces

**Day 2:**
- Topic: Kinematics
- Goals:
  - Learn about displacement, velocity, and acceleration
  - Practice using kinematic equations to solve problems related to motion
  - Understand the difference betwe

In [ ]:
# ── Interactive Mode: Chat with StudyBot yourself! ──────────────────
# Run this cell to start your own conversation
# Type 'quit' to exit

print("🎓 StudyBot — Interactive Mode")
print("=" * 50)
print("Type your message and press Enter. Type 'quit' to exit.\n")

my_history = []  # ← persistent history for this session

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q"]:
        print("\n👋 StudyBot: Good luck with your studies! You've got this! 🌟")
        break
    if not user_input:
        continue

    response = chat(user_input, history=my_history)
    print(f"\n🤖 StudyBot: {response}\n")

---
## 📖 Quick Reference — Everything We Covered

### The LangChain Building Blocks

```python
# 1️⃣  BASICS — Talk to an LLM
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")
response = llm.invoke("Your question")

# 2️⃣  PROMPTS — Reusable templates
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("Explain {topic} simply.")
messages = prompt.format_messages(topic="gravity")

# 3️⃣  CHAINS — Connect with | pipe
from langchain_core.output_parsers import StrOutputParser
chain = prompt | llm | StrOutputParser()
result = chain.invoke({"topic": "gravity"})

# 4️⃣  MEMORY — Remember conversations (LangChain 1.x)
# No special class needed — just maintain a messages list
chat_history = []
chat_history.append({"role": "user", "content": "hi"})
result = agent.invoke({"messages": chat_history})
reply = result["messages"][-1].content
chat_history.append({"role": "assistant", "content": reply})

# 5️⃣  TOOLS — Give LLM superpowers
from langchain_core.tools import tool
@tool
def my_tool(input: str) -> str:
    """Description the LLM reads to decide when to use this tool."""
    return "result"

# 6️⃣  AGENTS — LLM decides what to do (LangChain 1.x)
from langchain.agents import create_agent
agent = create_agent(model=llm, tools=tools, system_prompt="You are a helpful assistant.")
result = agent.invoke({"messages": [{"role": "user", "content": "question"}]})
print(result["messages"][-1].content)
```

---

## 🚀 What to Build Next

| Project | Concepts Used |
|---------|--------------|
| 📄 Document Q&A (RAG) | Chains + Vector stores |
| 🛒 E-commerce assistant | Agents + Memory + Tools |
| 📊 Data analyst bot | Tools + Structured output |
| 📅 Calendar assistant | Tools (Google Calendar API) |
| 🎮 Text adventure game | Memory + Chains + State |

---

### 🔗 Resources
- [LangChain Docs](https://python.langchain.com/docs/introduction)
- [LangChain GitHub](https://github.com/langchain-ai/langchain)
- [LangChain v1 Migration Guide](https://docs.langchain.com/oss/python/migrate/langchain-v1)
- [LangSmith](https://smith.langchain.com/) — debug and trace your chains
- [LangChain Hub](https://smith.langchain.com/hub) — community prompts